<a href="https://colab.research.google.com/github/ssstttaaasss/video_monitoring_system/blob/main/video_monitoring_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# CELL 1  |  DEPENDENCIES
# ================================================================

!pip install ultralytics opencv-python-headless gradio pillow matplotlib requests -q
print("Готово.")

Готово.


In [ ]:
# ================================================================
# CELL 2  |  SYSTEM CORE
# ================================================================

import cv2, numpy as np, time, csv, io, torch
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import requests as _req
from PIL         import Image, ImageDraw
from datetime    import datetime, timedelta
from typing      import Optional, List, Tuple
from ultralytics import YOLO

# ── Configuration ──────────────────────────────────────────────
CONFIG = {
    "model_path"             : "yolov8s.pt",
    "confidence_threshold"   : 0.35,
    "imgsz"                  : 960,
    "half_precision"         : torch.cuda.is_available(),
    "detection_classes"      : [0],
    "zone_alpha"             : 0.25,
    "zone_color_safe"        : (0, 200, 0),
    "zone_color_alert"       : (0, 0, 220),
    "zone_border_thickness"  : 2,
    "bbox_color_safe"        : (0, 255, 0),
    "bbox_color_alert"       : (0, 0, 255),
    "bbox_thickness"         : 2,
    "alert_border_color"     : (0, 0, 255),
    "alert_border_thickness" : 8,
    "hud_height"             : 40,
    "hud_bg_color"           : (15, 15, 15),
    "hud_text_color"         : (210, 210, 210),
    "hud_alert_color"        : (80, 80, 255),
    "video_codec"            : "mp4v",
    "output_video"           : "output.mp4",
    "output_log"             : "event_log.txt",
    "output_csv"             : "event_log.csv",
    "notif_cooldown_sec"     : 20,
}

model = YOLO(CONFIG["model_path"])
print(f"Model  : {CONFIG['model_path']}")
print(f"Device : {'GPU  (FP16)' if CONFIG['half_precision'] else 'CPU'}")


# ── ZoneManager ───────────────────────────────────────────────
class ZoneManager:
    """Polygon-based controlled zone with cv2 rendering."""

    def __init__(self, points: List[Tuple[int, int]], fw: int, fh: int):
        self.pts = np.array(points, dtype=np.int32)
        self.pts[:, 0] = np.clip(self.pts[:, 0], 0, fw - 1)
        self.pts[:, 1] = np.clip(self.pts[:, 1], 0, fh - 1)
        self.fw, self.fh = fw, fh

    def person_in_zone(self, x1: int, y1: int, x2: int, y2: int) -> bool:
        fx, fy = (x1 + x2) // 2, y2
        return cv2.pointPolygonTest(self.pts, (float(fx), float(fy)), False) >= 0

    def draw(self, frame: np.ndarray, alert: bool) -> np.ndarray:
        color = CONFIG["zone_color_alert"] if alert else CONFIG["zone_color_safe"]
        ov = frame.copy()
        cv2.fillPoly(ov, [self.pts], color)
        cv2.addWeighted(ov, CONFIG["zone_alpha"], frame, 1 - CONFIG["zone_alpha"], 0, frame)
        cv2.polylines(frame, [self.pts], True, color, CONFIG["zone_border_thickness"])
        lx = int(self.pts[:, 0].min())
        ly = max(int(self.pts[:, 1].min()) - 10, 14)
        cv2.putText(frame, "! DANGER ZONE" if alert else "CONTROLLED ZONE",
                    (lx, ly), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)
        return frame

    @property
    def info(self) -> dict:
        area = float(cv2.contourArea(self.pts))
        return {
            "points"      : self.pts.tolist(),
            "area_pixels" : round(area),
            "area_percent": round(area / (self.fw * self.fh) * 100, 2),
        }


# ── EventLogger ───────────────────────────────────────────────
class EventLogger:
    """Records zone breach transitions and computes session statistics."""

    BREACH = "BREACH"
    CLEAR  = "CLEAR"

    def __init__(self, fps: float):
        self.fps    = fps
        self.events : List[dict] = []
        self._last  = False

    def _ts(self, idx: int) -> str:
        td = timedelta(seconds=idx / self.fps)
        h, r = divmod(td.seconds, 3600)
        m, s = divmod(r, 60)
        return f"{h:02d}:{m:02d}:{s:02d}.{td.microseconds // 1000:03d}"

    def update(self, idx: int, alert: bool, count: int):
        if alert != self._last:
            self.events.append({
                "type"     : self.BREACH if alert else self.CLEAR,
                "frame"    : idx,
                "timestamp": self._ts(idx),
                "in_zone"  : count,
            })
            self._last = alert

    def statistics(self, total: int) -> dict:
        entries = [e for e in self.events if e["type"] == self.BREACH]
        exits   = [e for e in self.events if e["type"] == self.CLEAR]
        af = sum(
            (exits[i]["frame"] if i < len(exits) else total) - e["frame"]
            for i, e in enumerate(entries)
        )
        return {
            "всього_кадрів"    : total,
            "тривалість_сек"   : round(total / self.fps, 2),
            "кількість_тривог" : len(entries),
            "час_тривоги_сек"  : round(af / self.fps, 2),
            "відсоток_тривоги" : round(af / max(total, 1) * 100, 2),
        }

    def export_txt(self, path: str):
        with open(path, "w", encoding="utf-8") as f:
            f.write("=" * 64 + "\n")
            f.write("  СИСТЕМА ВІДЕОМОНІТОРИНГУ — ЖУРНАЛ ПОДІЙ\n")
            f.write(f"  {datetime.now().strftime('%Y-%m-%d  %H:%M:%S')}\n")
            f.write("=" * 64 + "\n\n")
            if not self.events:
                f.write("Подій не зафіксовано.\n")
                return
            for e in self.events:
                arrow = ">>> ЗОНА ПОРУШЕНА" if e["type"] == self.BREACH else "<<< ЗОНА ВІЛЬНА "
                f.write(f"[{e['timestamp']}]  {arrow}  | у зоні: {e['in_zone']}\n")

    def export_csv(self, path: str):
        with open(path, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=["type", "frame", "timestamp", "in_zone"])
            w.writeheader()
            w.writerows(self.events)


# ── Notifiers ─────────────────────────────────────────────────
class TelegramNotifier:
    """
    Sends breach/clear alerts via Telegram Bot API.
    Messages include camera name, timestamp, and event type.
    Photos of breach frames are sent automatically.
    Cooldown prevents notification spam.
    """

    def __init__(self, token: str, chat_id: str,
                 cooldown_sec: int = CONFIG["notif_cooldown_sec"]):
        self.token      = token.strip()
        self.chat_id    = chat_id.strip()
        self.cooldown   = cooldown_sec
        self._last_sent = 0.0
        self._api       = f"https://api.telegram.org/bot{self.token}"

    def _ready(self) -> bool:
        return time.time() - self._last_sent >= self.cooldown

    def send_text(self, text: str):
        if not self._ready():
            return
        try:
            _req.post(
                f"{self._api}/sendMessage",
                json    = {"chat_id": self.chat_id, "text": text, "parse_mode": "HTML"},
                timeout = 5,
            )
            self._last_sent = time.time()
        except Exception:
            pass

    def send_photo(self, caption: str, frame_bgr: np.ndarray):
        if not self._ready():
            return
        try:
            _, buf = cv2.imencode(".jpg", frame_bgr, [cv2.IMWRITE_JPEG_QUALITY, 82])
            _req.post(
                f"{self._api}/sendPhoto",
                files   = {"photo": ("alert.jpg", buf.tobytes(), "image/jpeg")},
                data    = {"chat_id": self.chat_id, "caption": caption, "parse_mode": "HTML"},
                timeout = 10,
            )
            self._last_sent = time.time()
        except Exception:
            pass


class WebhookNotifier:
    """
    Generic HTTP POST webhook.
    Compatible with Discord, Slack, and any custom endpoint.
    """

    def __init__(self, url: str, cooldown_sec: int = CONFIG["notif_cooldown_sec"]):
        self.url        = url.strip()
        self.cooldown   = cooldown_sec
        self._last_sent = 0.0

    def send_text(self, text: str):
        if time.time() - self._last_sent < self.cooldown:
            return
        try:
            _req.post(self.url, json={"content": text, "text": text}, timeout=5)
            self._last_sent = time.time()
        except Exception:
            pass

    def send_photo(self, caption: str, frame_bgr: np.ndarray):
        self.send_text(caption)


# ── HUD renderer ──────────────────────────────────────────────
def _draw_hud(frame: np.ndarray, idx: int, fps: float,
              total: int, in_zone: int, alert: bool,
              camera_name: str = "Камера 1") -> np.ndarray:
    h, w = frame.shape[:2]
    ov   = frame.copy()
    cv2.rectangle(ov, (0, 0), (w, CONFIG["hud_height"]), CONFIG["hud_bg_color"], -1)
    cv2.addWeighted(ov, 0.78, frame, 0.22, 0, frame)

    ts  = idx / fps
    tss = f"{int(ts // 60):02d}:{ts % 60:05.2f}"
    cv2.putText(frame, tss,                    (10,  27), cv2.FONT_HERSHEY_SIMPLEX, 0.60, CONFIG["hud_text_color"], 1, cv2.LINE_AA)
    cv2.putText(frame, f"FRAME {idx:05d}",     (135, 27), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (130, 130, 130),          1, cv2.LINE_AA)
    cv2.putText(frame, f"DET: {total}",        (265, 27), cv2.FONT_HERSHEY_SIMPLEX, 0.52, CONFIG["hud_text_color"], 1, cv2.LINE_AA)
    cv2.putText(frame, camera_name,            (340, 27), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (160, 160, 160),          1, cv2.LINE_AA)

    if alert:
        bc = CONFIG["hud_alert_color"] if (idx // 8) % 2 == 0 else (200, 200, 200)
        cv2.putText(frame, f"!! ALERT  IN ZONE: {in_zone} !!", (w - 365, 27),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.62, bc, 2, cv2.LINE_AA)
    else:
        cv2.putText(frame, "ZONE CLEAR", (w - 195, 27),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.58, (0, 200, 0), 1, cv2.LINE_AA)
    return frame


# ── Frame processor ───────────────────────────────────────────
def process_frame(frame: np.ndarray, idx: int, fps: float,
                  zone_mgr: ZoneManager, logger: EventLogger,
                  camera_name: str = "Камера 1") -> Tuple[np.ndarray, bool]:

    results = model(
        frame,
        conf    = CONFIG["confidence_threshold"],
        classes = CONFIG["detection_classes"],
        imgsz   = CONFIG["imgsz"],
        half    = CONFIG["half_precision"],
        verbose = False,
    )
    boxes   = results[0].boxes
    in_zone = sum(1 for b in boxes
                  if zone_mgr.person_in_zone(*[int(v) for v in b.xyxy[0].tolist()]))
    alert   = in_zone > 0

    frame = zone_mgr.draw(frame, alert)

    for box in boxes:
        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
        conf  = float(box.conf[0])
        in_z  = zone_mgr.person_in_zone(x1, y1, x2, y2)
        color = CONFIG["bbox_color_alert"] if in_z else CONFIG["bbox_color_safe"]
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, CONFIG["bbox_thickness"])
        lbl = f"person {conf:.2f}"
        (lw, lh), _ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.50, 1)
        cv2.rectangle(frame, (x1, y1 - lh - 6), (x1 + lw + 4, y1), color, -1)
        cv2.putText(frame, lbl, (x1 + 2, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 1, cv2.LINE_AA)
        cv2.circle(frame, ((x1 + x2) // 2, y2), 4, color, -1)

    if alert:
        fh, fw = frame.shape[:2]
        cv2.rectangle(frame, (0, 0), (fw - 1, fh - 1),
                      CONFIG["alert_border_color"], CONFIG["alert_border_thickness"])

    frame = _draw_hud(frame, idx, fps, len(boxes), in_zone, alert, camera_name)
    logger.update(idx, alert, in_zone)
    return frame, alert


# ── Video pipeline ────────────────────────────────────────────
def process_video(
    video_path  : str,
    zone_mgr    : ZoneManager,
    output_path : str,
    log_path    : str,
    csv_path    : str,
    camera_name : str           = "Камера 1",
    max_frames  : Optional[int] = None,
    progress_cb                 = None,
    notifiers   : list          = [],
) -> Tuple[dict, EventLogger]:

    cap    = cv2.VideoCapture(video_path)
    tot    = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps    = cap.get(cv2.CAP_PROP_FPS)
    fw     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    fh     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    lim    = min(tot, max_frames) if max_frames else tot
    fourcc = cv2.VideoWriter_fourcc(*CONFIG["video_codec"])
    writer = cv2.VideoWriter(output_path, fourcc, fps, (fw, fh))
    logger = EventLogger(fps)

    t0          = time.time()
    idx         = 0
    prev_alert  = False

    while idx < lim:
        ret, frame = cap.read()
        if not ret:
            break

        ann, alert = process_frame(frame, idx, fps, zone_mgr, logger, camera_name)
        writer.write(ann)

        # Notify on breach start (transition only, not every frame)
        if alert and not prev_alert and notifiers:
            ts_str = str(timedelta(seconds=idx / fps)).split(".")[0]
            msg = (
                f"🚨 <b>ТРИВОГА — ЗОНА ПОРУШЕНА</b>\n"
                f"📷 Камера: <b>{camera_name}</b>\n"
                f"⏱ Час відео: {ts_str}\n"
                f"🎞 Кадр: {idx}"
            )
            for n in notifiers:
                n.send_photo(msg, frame)

        # Notify on zone clear
        if not alert and prev_alert and notifiers:
            ts_str = str(timedelta(seconds=idx / fps)).split(".")[0]
            msg = (
                f"✅ <b>Зона вільна</b>\n"
                f"📷 Камера: <b>{camera_name}</b>\n"
                f"⏱ Час відео: {ts_str}"
            )
            for n in notifiers:
                n.send_text(msg)

        prev_alert = alert
        idx += 1
        if progress_cb and idx % 15 == 0:
            progress_cb(idx / lim, f"Кадр {idx} / {lim}")

    cap.release()
    writer.release()
    logger.export_txt(log_path)
    logger.export_csv(csv_path)

    stats = logger.statistics(idx)
    stats["час_обробки_сек"] = round(time.time() - t0, 2)
    return stats, logger


# ── Zone preview (PIL) ────────────────────────────────────────
def draw_zone_preview(frame_bgr: np.ndarray, points: list) -> np.ndarray:
    """Returns RGB numpy array with interactive zone polygon for Gradio display."""
    img = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)).convert("RGBA")
    ov  = Image.new("RGBA", img.size, (0, 0, 0, 0))
    d   = ImageDraw.Draw(ov)
    if len(points) >= 3:
        flat = [coord for pt in points for coord in pt]
        d.polygon(flat, fill=(0, 200, 0, 55), outline=(0, 200, 0, 220))
    elif len(points) == 2:
        d.line([tuple(points[0]), tuple(points[1])], fill=(0, 200, 0, 220), width=2)
    img = Image.alpha_composite(img, ov).convert("RGB")
    d2  = ImageDraw.Draw(img)
    for i, (x, y) in enumerate(points):
        r = 7
        d2.ellipse([x - r, y - r, x + r, y + r],
                   fill=(0, 255, 255), outline=(255, 255, 255), width=2)
        d2.text((x + 11, y - 9), str(i + 1), fill=(0, 255, 255))
    return np.array(img)


# ── Alert timeline chart ──────────────────────────────────────
def make_alert_chart(logger: EventLogger, total_frames: int) -> np.ndarray:
    """Generates matplotlib alert timeline, returns RGB numpy array."""
    fps     = logger.fps
    dur     = total_frames / fps
    entries = [e for e in logger.events if e["type"] == EventLogger.BREACH]
    exits   = [e for e in logger.events if e["type"] == EventLogger.CLEAR]

    fig, ax = plt.subplots(figsize=(11, 2.5), facecolor="#1a1a1a")
    ax.set_facecolor("#111111")
    ax.set_xlim(0, dur)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Час (секунди)", color="#999", fontsize=9)
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_edgecolor("#333")
    ax.tick_params(colors="#999")
    ax.axhspan(0, 1, color="#152215", alpha=1.0)

    for i, entry in enumerate(entries):
        t0_ = entry["frame"] / fps
        t1_ = exits[i]["frame"] / fps if i < len(exits) else dur
        ax.axvspan(t0_, t1_, color="#cc2222", alpha=0.75)

    ax.set_title("Часова шкала тривог", color="#ddd", fontsize=10, pad=5)
    ax.legend(
        handles=[
            mpatches.Patch(color="#cc2222", alpha=0.75, label="Тривога"),
            mpatches.Patch(color="#152215", alpha=1.0,  label="Зона вільна"),
        ],
        loc="upper right", facecolor="#1a1a1a",
        labelcolor="#ccc", fontsize=8, framealpha=0.9,
    )
    plt.tight_layout(pad=0.4)
    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=110, facecolor="#1a1a1a")
    plt.close(fig)
    buf.seek(0)
    return np.array(Image.open(buf))


print("Система ініціалізована.")

Model  : yolov8s.pt
Device : GPU  (FP16)
Система ініціалізована.


In [ ]:
# ================================================================
# CELL 3  |  GRADIO INTERFACE
# ================================================================

import gradio as gr


# ── Utilities ─────────────────────────────────────────────────
def _video_meta(path: str) -> dict:
    cap = cv2.VideoCapture(path)
    m = {
        "width"  : int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height" : int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "fps"    : cap.get(cv2.CAP_PROP_FPS),
        "frames" : int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    }
    m["duration"] = round(m["frames"] / m["fps"], 1)
    cap.release()
    return m


def _get_frame(path: str, idx: int = 60) -> Optional[np.ndarray]:
    cap = cv2.VideoCapture(path)
    tot = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, min(idx, tot - 1))
    ret, frame = cap.read()
    cap.release()
    return frame if ret else None


def _build_notifiers(enabled, tg_token, tg_chat, webhook_url):
    """Constructs active notifier list from UI inputs."""
    notifiers = []
    if enabled:
        if tg_token.strip() and tg_chat.strip():
            notifiers.append(TelegramNotifier(tg_token.strip(), tg_chat.strip()))
        if webhook_url.strip():
            notifiers.append(WebhookNotifier(webhook_url.strip()))
    return notifiers


# ── Event handlers ─────────────────────────────────────────────
def on_upload(video_path):
    if video_path is None:
        return None, [], "", ""
    m     = _video_meta(video_path)
    frame = _get_frame(video_path)
    rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    info  = (f"Роздільність: {m['width']} × {m['height']} пкс  |  "
             f"FPS: {m['fps']:.1f}  |  Тривалість: {m['duration']} с  |  "
             f"X: 0–{m['width']}   Y: 0–{m['height']}")
    return rgb, [], "", info


def on_click(evt: gr.SelectData, points: list, video_path: str):
    if video_path is None:
        return None, points, ""
    pts    = points + [[int(evt.index[0]), int(evt.index[1])]]
    frame  = _get_frame(video_path)
    prev   = draw_zone_preview(frame, pts)
    coords = "\n".join(f"P{i+1}: ({p[0]}, {p[1]})" for i, p in enumerate(pts))
    return prev, pts, coords


def on_undo(points: list, video_path: str):
    frame = _get_frame(video_path) if video_path else None
    if frame is None:
        return None, [], ""
    pts    = points[:-1] if points else []
    prev   = draw_zone_preview(frame, pts) if pts else cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    coords = "\n".join(f"P{i+1}: ({p[0]}, {p[1]})" for i, p in enumerate(pts))
    return prev, pts, coords


def on_clear(video_path: str):
    frame = _get_frame(video_path) if video_path else None
    if frame is None:
        return None, [], ""
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), [], ""


def on_manual_apply(manual_text: str, video_path: str):
    if video_path is None:
        return None, [], "", "Спочатку завантажте відео."
    try:
        pts = []
        for pair in manual_text.replace(" ", "").split(";"):
            if pair:
                x, y = pair.split(",")
                pts.append([int(x), int(y)])
        if len(pts) < 3:
            return None, [], "", "Потрібно мінімум 3 точки."
        frame  = _get_frame(video_path)
        prev   = draw_zone_preview(frame, pts)
        coords = "\n".join(f"P{i+1}: ({p[0]}, {p[1]})" for i, p in enumerate(pts))
        return prev, pts, coords, "✓ Координати застосовано."
    except Exception as e:
        return None, [], "", f"Помилка: {e}"


def on_toggle_token(v_pwd, v_txt, showing):
    """Toggles Telegram token between masked and visible mode."""
    showing = not showing
    val     = (v_txt if showing else v_pwd).strip()
    if showing:
        return (
            gr.update(visible=False, value=val),
            gr.update(visible=True,  value=val),
            showing,
            gr.update(value="🔒"),
        )
    else:
        return (
            gr.update(visible=True,  value=val),
            gr.update(visible=False, value=val),
            showing,
            gr.update(value="👁"),
        )


def on_test_notification(enabled, tg_token_p, tg_token_t, tg_chat,
                         webhook_url, camera_name):
    """Sends a test message through all configured notifiers."""
    token      = (tg_token_t or tg_token_p).strip()
    notifiers  = _build_notifiers(enabled, token, tg_chat, webhook_url)
    if not notifiers:
        return "⚠ Увімкніть сповіщення та налаштуйте хоча б один канал."
    cam = camera_name.strip() or "Камера 1"
    msg = (
        f"✅ <b>Тестове сповіщення</b>\n"
        f"📷 Камера: <b>{cam}</b>\n"
        f"🕐 {datetime.now().strftime('%H:%M:%S')}\n"
        f"Система відеомоніторингу активна."
    )
    for n in notifiers:
        n.send_text(msg)
    return "✓ Тестове сповіщення надіслано."


def on_run(video_path, points, confidence, max_f,
           camera_name, notif_enabled, tg_token_p, tg_token_t,
           tg_chat, webhook_url,
           progress=gr.Progress()):

    if video_path is None:
        raise gr.Error("Завантажте відеофайл.")
    if len(points) < 3:
        raise gr.Error("Визначте зону — мінімум 3 точки.")

    CONFIG["confidence_threshold"] = float(confidence)
    m    = _video_meta(video_path)
    zone = ZoneManager([(p[0], p[1]) for p in points], m["width"], m["height"])
    lim  = int(max_f) if int(max_f) > 0 else None
    cam  = camera_name.strip() or "Камера 1"

    token     = (tg_token_t or tg_token_p).strip()
    notifiers = _build_notifiers(notif_enabled, token, tg_chat, webhook_url)

    stats, logger = process_video(
        video_path,
        zone_mgr    = zone,
        output_path = CONFIG["output_video"],
        log_path    = CONFIG["output_log"],
        csv_path    = CONFIG["output_csv"],
        camera_name = cam,
        max_frames  = lim,
        progress_cb = lambda v, d: progress(v, desc=d),
        notifiers   = notifiers,
    )

    with open(CONFIG["output_log"], encoding="utf-8") as f:
        log_text = f.read()

    chart = make_alert_chart(logger, stats["всього_кадрів"])

    summary = (
        f"ЗВІТ АНАЛІЗУ\n"
        f"{'─' * 52}\n"
        f"Дата:                {datetime.now().strftime('%Y-%m-%d  %H:%M:%S')}\n"
        f"Камера:              {cam}\n"
        f"Роздільність:        {m['width']} × {m['height']} пкс\n"
        f"FPS:                 {m['fps']:.1f}\n"
        f"{'─' * 52}\n"
        f"Кадрів оброблено:    {stats['всього_кадрів']}\n"
        f"Тривалість:          {stats['тривалість_сек']} сек\n"
        f"{'─' * 52}\n"
        f"Кількість тривог:    {stats['кількість_тривог']}\n"
        f"Час тривоги:         {stats['час_тривоги_сек']} сек\n"
        f"Відсоток тривоги:    {stats['відсоток_тривоги']} %\n"
        f"{'─' * 52}\n"
        f"Зона ({len(points)} точок):   "
        f"{', '.join(f'({p[0]},{p[1]})' for p in points)}\n"
        f"{'─' * 52}\n"
    )

    return (
        CONFIG["output_video"],
        stats,
        log_text,
        chart,
        summary,
        CONFIG["output_log"],
        CONFIG["output_csv"],
    )


# ── Layout ────────────────────────────────────────────────────
CSS = """
footer { display: none !important; }
.small-btn button { min-height: 36px !important; font-size: 0.82rem !important; }
.icon-btn button  { min-height: 36px !important; min-width: 44px !important;
                    font-size: 1.1rem !important; padding: 0 10px !important; }
.run-status { min-height: 28px; font-size: 0.88rem; }
"""

with gr.Blocks(
    title  = "Video Monitoring System",
    theme  = gr.themes.Base(
        primary_hue = "red",
        neutral_hue = "slate",
        font        = gr.themes.GoogleFont("Inter"),
    ),
    css = CSS,
) as demo:

    gr.Markdown("## VIDEO MONITORING SYSTEM")

    # Shared state
    points_state    = gr.State([])
    _tok_show_state = gr.State(False)

    with gr.Row(equal_height=False):

        # ── LEFT ──────────────────────────────────────────────
        with gr.Column(scale=1, min_width=315):

            gr.Markdown("#### Відеофайл")
            video_input  = gr.Video(label="Завантажити .mp4")
            camera_name  = gr.Textbox(
                label="Назва камери",
                value="Камера 1",
                placeholder="Камера 1",
                info="Відображається у HUD та сповіщеннях Telegram.",
            )
            video_info   = gr.Textbox(
                label="Властивості відео", interactive=False, lines=2,
                placeholder="Завантажте відео...",
            )

            gr.Markdown("#### Визначення зони")
            gr.Markdown(
                "<small>Клацайте на кадрі щоб додавати точки. Мінімум 3 точки.</small>"
            )
            zone_frame = gr.Image(
                label="Кадр — клацніть для позначення зони",
                interactive=False,
                show_download_button=False,
            )
            with gr.Row(elem_classes="small-btn"):
                btn_undo  = gr.Button("↩ Скасувати")
                btn_clear = gr.Button("✕ Очистити")

            coords_display = gr.Textbox(
                label="Точки зони", interactive=False, lines=4,
                placeholder="Точки з'являться після кліків...",
            )

            with gr.Accordion("Ввести координати вручну", open=False):
                gr.Markdown("<small>Формат: `X1,Y1;X2,Y2;X3,Y3;X4,Y4`</small>")
                manual_input  = gr.Textbox(
                    label="Координати",
                    placeholder="370,185;740,185;910,530;200,530",
                )
                manual_status = gr.Textbox(label="Статус", interactive=False, lines=1)
                btn_manual    = gr.Button("Застосувати", size="sm")

        # ── RIGHT ─────────────────────────────────────────────
        with gr.Column(scale=2):

            with gr.Tabs():

                # ── Tab: Аналіз ───────────────────────────────
                with gr.Tab("Аналіз"):
                    gr.Markdown("#### Параметри обробки")
                    confidence_slider = gr.Slider(
                        0.10, 0.90, value=0.35, step=0.05,
                        label="Поріг впевненості детекції",
                        info="Нижче = більше виявлень. Вище = менше хибних тривог.",
                    )
                    max_frames_num = gr.Number(
                        label="Максимум кадрів  (0 = повне відео)",
                        value=0, precision=0, minimum=0,
                        info="Введіть 200 для швидкого тесту.",
                    )
                    btn_run    = gr.Button("▶  Запустити аналіз", variant="primary", size="lg")
                    run_status = gr.Markdown("", elem_classes="run-status")

                # ── Tab: Відеорезультат ───────────────────────
                with gr.Tab("Відеорезультат"):
                    output_video = gr.Video(
                        label="Оброблене відео",
                        show_download_button=True,
                    )

                # ── Tab: Статистика ───────────────────────────
                with gr.Tab("Статистика"):
                    alert_chart = gr.Image(
                        label="Часова шкала тривог",
                        show_download_button=True,
                    )
                    stats_out = gr.JSON(label="Зведена статистика")

                # ── Tab: Журнал подій ─────────────────────────
                with gr.Tab("Журнал подій"):
                    log_out = gr.Textbox(
                        label="Журнал", lines=18, show_copy_button=True,
                        placeholder="Журнал з'явиться після аналізу...",
                    )
                    with gr.Row():
                        dl_txt = gr.File(label="Завантажити .txt")
                        dl_csv = gr.File(label="Завантажити .csv")

                # ── Tab: Звіт ─────────────────────────────────
                with gr.Tab("Звіт"):
                    summary_out = gr.Textbox(
                        label="Текстовий звіт",
                        lines=18,
                        show_copy_button=True,
                        placeholder="Результати аналізу з'являться після обробки відео...",
                    )

                # ── Tab: Сповіщення ───────────────────────────
                with gr.Tab("Сповіщення"):

                    notif_enabled = gr.Checkbox(
                        label="Увімкнути сповіщення",
                        value=False,
                        info="Вмикає відправку повідомлень під час аналізу відео.",
                    )

                    gr.Markdown("---")
                    gr.Markdown("#### Telegram")
                    gr.Markdown(
                        "<small>"
                        "1. Відкрийте [@BotFather](https://t.me/BotFather) → `/newbot` → отримайте токен.<br>"
                        "2. Напишіть боту будь-яке повідомлення.<br>"
                        "3. Відкрийте `https://api.telegram.org/bot<ТОКЕН>/getUpdates` у браузері.<br>"
                        "4. Знайдіть `\"chat\": {\"id\": XXXXXX}` — це ваш Chat ID."
                        "</small>"
                    )

                    # Token field with show/hide button
                    with gr.Row():
                        tg_token_pwd = gr.Textbox(
                            label="Bot Token",
                            type="password",
                            placeholder="1234567890:ABCdef...",
                            scale=9,
                        )
                        tg_token_txt = gr.Textbox(
                            label="Bot Token",
                            type="text",
                            placeholder="1234567890:ABCdef...",
                            scale=9,
                            visible=False,
                        )
                        btn_tok_vis = gr.Button(
                            "👁", size="sm", scale=1,
                            elem_classes="icon-btn",
                        )

                    tg_chat = gr.Textbox(
                        label="Chat ID",
                        placeholder="-100123456789",
                    )

                    gr.Markdown("---")
                    gr.Markdown("#### Webhook (Discord / Slack / інше)")
                    webhook_url = gr.Textbox(
                        label="Webhook URL",
                        placeholder="https://discord.com/api/webhooks/...",
                    )

                    gr.Markdown("---")
                    with gr.Row():
                        btn_test_notif = gr.Button("Надіслати тест", size="sm")
                        notif_status   = gr.Textbox(
                            label="Статус", interactive=False,
                            lines=1, scale=3,
                        )

    # ── Bindings ───────────────────────────────────────────────

    video_input.change(
        fn      = on_upload,
        inputs  = [video_input],
        outputs = [zone_frame, points_state, coords_display, video_info],
    )
    zone_frame.select(
        fn      = on_click,
        inputs  = [points_state, video_input],
        outputs = [zone_frame, points_state, coords_display],
    )
    btn_undo.click(
        fn      = on_undo,
        inputs  = [points_state, video_input],
        outputs = [zone_frame, points_state, coords_display],
    )
    btn_clear.click(
        fn      = on_clear,
        inputs  = [video_input],
        outputs = [zone_frame, points_state, coords_display],
    )
    btn_manual.click(
        fn      = on_manual_apply,
        inputs  = [manual_input, video_input],
        outputs = [zone_frame, points_state, coords_display, manual_status],
    )

    # Token show/hide toggle
    btn_tok_vis.click(
        fn      = on_toggle_token,
        inputs  = [tg_token_pwd, tg_token_txt, _tok_show_state],
        outputs = [tg_token_pwd, tg_token_txt, _tok_show_state, btn_tok_vis],
    )

    # Test notification
    btn_test_notif.click(
        fn      = on_test_notification,
        inputs  = [notif_enabled, tg_token_pwd, tg_token_txt, tg_chat,
                   webhook_url, camera_name],
        outputs = [notif_status],
    )

    # Run analysis — chained for immediate status feedback
    btn_run.click(
        fn      = lambda: "⏳  Аналіз виконується...",
        inputs  = [],
        outputs = [run_status],
    ).then(
        fn      = on_run,
        inputs  = [
            video_input, points_state, confidence_slider, max_frames_num,
            camera_name, notif_enabled, tg_token_pwd, tg_token_txt,
            tg_chat, webhook_url,
        ],
        outputs = [output_video, stats_out, log_out, alert_chart,
                   summary_out, dl_txt, dl_csv],
    ).then(
        fn      = lambda: "✅  Готово. Перейдіть до вкладки «Відеорезультат».",
        inputs  = [],
        outputs = [run_status],
    )


print("Інтерфейс готовий.")

/tmp/ipykernel_1300/3481434366.py:216: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_1300/3481434366.py:216: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


Інтерфейс готовий.


In [ ]:
# ================================================================
# CELL 4  |  LAUNCH
# ================================================================

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://769e9d0a85c16ae825.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
